In [1]:
# Load environment variables and suppress warnings for clean output
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-5-mini", temperature=0)
parser = StrOutputParser()

print("Model initialized with gpt-5-mini.")

Model initialized with gpt-5-mini.


In [3]:
# Define modular prompt components as reusable building blocks
system_block = "You are a technical support assistant. Keep output short and structured."

format_block = (
    "Return exactly three lines:\n"
    "Severity: <Low|Medium|High>\n"
    "Summary: <one line>\n"
    "Next: <one action>"
)

task_block = (
    "Customer: {customer}\n"
    "Product area: {product_area}\n"
    "Platform: {platform}\n\n"
    "Symptoms: {symptoms}\n"
    "Timeline: {timeline}\n\n"
    "{format_block}"
)

print("Modular blocks created.")

Modular blocks created.


In [ ]:
# Compose modular blocks into a single ChatPromptTemplate
# The `partial` method allows for dynamic insertion of the `format_block` into the `task_block` at runtime.
# partial() makes a reusable prompt template cleaner by fixing some variables early and leaving only request-specific values to be supplied later.
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_block),
    ("human", task_block),
]).partial(format_block=format_block)

print("ChatPromptTemplate composed from modular blocks.")

ChatPromptTemplate composed from modular blocks.


In [5]:
# Define realistic application input
input_data = {
    "customer": "Northwind Traders",
    "product_area": "Dashboard",
    "platform": "Windows laptop",
    "symptoms": "Loads in 20–30 seconds, sometimes TIMEOUT_504",
    "timeline": "Started after yesterday's update",
}

print("Input data ready.")

Input data ready.


In [6]:
# Build end-to-end chain using LCEL: prompt -> model -> parser
chain = chat_prompt | model | parser

result = chain.invoke(input_data)

print("=== END TO END CHAIN OUTPUT ===")
print(result)

=== END TO END CHAIN OUTPUT ===
Severity: High
Summary: Dashboard on Windows laptops began loading slowly (20–30s) and intermittently returning TIMEOUT_504 after yesterday's update.
Next: Roll back the Dashboard update deployed yesterday on affected Windows machines and verify load times and 504 errors.


In [7]:
# Swap only the output format block to show mix-and-match modularity
format_block_detailed = (
    "Return exactly five lines:\n"
    "Severity: <Low|Medium|High>\n"
    "Summary: <one line>\n"
    "Root cause hypothesis: <one line>\n"
    "Next: <one action>\n"
    "ETA: <hours or days>"
)

chain_detailed = chat_prompt.partial(format_block=format_block_detailed) | model | parser

result_detailed = chain_detailed.invoke(input_data)

print("=== DETAILED FORMAT OUTPUT ===")
print(result_detailed)

=== DETAILED FORMAT OUTPUT ===
Severity: High
Summary: Dashboard loads slowly (20–30s) and intermittently returns TIMEOUT_504 after yesterday's update.
Root cause hypothesis: Recent update introduced a backend regression causing slow DB queries or API gateway timeouts.
Next: Collect backend (API/gateway) logs and APM traces for failing requests during the incidents.
ETA: 4 hours


In [8]:
print(type(chat_prompt))
print(type(chain))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
<class 'langchain_core.runnables.base.RunnableSequence'>
